In [1]:
##-------------------------------------------------------------------------------------------------------
# SCRIPT TO RUN QUAL DATA CLINICAL MODELS
#--------------------------------------------------------------------------------------------------------
import pandas as pd
import numpy as np
import torch
import random
import matplotlib.pyplot as plt
import seaborn as sns
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import KFold, train_test_split
from sklearn.metrics import mean_squared_error, accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from tqdm.auto import tqdm
from sklearn.metrics import cohen_kappa_score, f1_score, recall_score, confusion_matrix

# -----------------------------
# IMPORT DATA CLEANED
# -----------------------------
data = pd.read_csv('Comments_Cleaned.csv',encoding='latin-1')
target_cols = ["AVERAGE EVIDENCE", "AVERAGE SUGGESTION", "AVERAGE CONNECTION"]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print('Using Device:', device)


/blue/prismap-ai-core/vnolan/conda/envs/APARI2_Models/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using Device: cuda


In [2]:
# -----------------------------
# CONVERT EVIDENCE TO BINARY
# -----------------------------
data["EVIDENCE1"] = (data["EVIDENCE1"] >= 2).astype(int)
data["EVIDENCE2"] = (data["EVIDENCE2"] >= 2).astype(int)


In [3]:
# Recalculate mQuAL totals on the 0–3 scale
data["TOTAL1"] = (
    data["EVIDENCE1"]
    + data["SUGGESTION1"]
    + data["CONNECTION1"]
)

data["TOTAL2"] = (
    data["EVIDENCE2"]
    + data["SUGGESTION2"]
    + data["CONNECTION2"]
)

# Optional validation
assert data["TOTAL1"].between(0, 3).all()
assert data["TOTAL2"].between(0, 3).all()

print("Rater 1 mQuAL range:", data["TOTAL1"].min(), data["TOTAL1"].max())
print("Rater 2 mQuAL range:", data["TOTAL2"].min(), data["TOTAL2"].max())

Rater 1 mQuAL range: 0 3
Rater 2 mQuAL range: 0 3


In [4]:
# -----------------------------
# EXPLORING DATA
# -----------------------------
print('Number of Comments:', len(data))
disagreements_evidence = (data["EVIDENCE1"] != data["EVIDENCE2"]).sum()
print('Evidence Disagreements:', disagreements_evidence)
disagreements_suggestion = (data["SUGGESTION1"] != data["SUGGESTION2"]).sum()
print('Suggestion Disagreements:', disagreements_suggestion)
disagreements_connection = (data["CONNECTION1"] != data["CONNECTION2"]).sum()
print('Connection Disagreements:', disagreements_connection)

## INTER-RATER RELIABILITY ##
# Evidence (binary)
kappa_evidence = cohen_kappa_score(data["EVIDENCE1"], data["EVIDENCE2"])
# Suggestion (binary)
kappa_suggestion = cohen_kappa_score(data["SUGGESTION1"], data["SUGGESTION2"])
# Connection (binary)
kappa_connection = cohen_kappa_score(data["CONNECTION1"], data["CONNECTION2"])
print('InterRater Reliability Scores:', kappa_evidence, kappa_suggestion, kappa_connection)

## NUMBER OF ROWS OF DISAGREEMENT
criteria = ["EVIDENCE", "SUGGESTION", "CONNECTION"]
disagreement = False
for c in criteria:
    disagreement |= data[f"{c}1"] != data[f"{c}2"]
rows_to_drop = data[disagreement].shape[0]

## DROPPING ROWS WITH ANY DISAGREEMENT
data = data[~disagreement].copy()
print('No. Comments with No Disagreement:', len(data))

# -----------------------------
# CHECK LENGTH OF COMMENTS TOKENIZED
# -----------------------------
MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
data["token_count"] = data["COMMENTS"].apply(lambda x: len(tokenizer.encode(str(x), add_special_tokens=True)))
print('Max Tokens:', data["token_count"].max())

#------------------------------------
# SPLIT AND COLLAPSE DATA
#------------------------------------
unique_comments = data["COMMENTS"].unique()
train_comments, temp_comments = train_test_split(unique_comments, test_size=0.20, random_state=42)
val_comments, test_comments = train_test_split(temp_comments, test_size=0.50, random_state=42)


train_data = data[data["COMMENTS"].isin(train_comments)]
val_data   = data[data["COMMENTS"].isin(val_comments)]
test_data  = data[data["COMMENTS"].isin(test_comments)]


print("Train:", len(train_comments), len(train_comments) / len(unique_comments))
print("Validation:", len(val_comments), len(val_comments) / len(unique_comments))
print("Test:", len(test_comments), len(test_comments) / len(unique_comments))


# Collapse each split independently
def collapse(df):
    df_r1 = df[['COMMENTS','EVIDENCE1','SUGGESTION1','CONNECTION1','TOTAL1']].copy()
    df_r1['RATER'] = 1
    df_r1.columns = ['COMMENTS','EVIDENCE','SUGGESTION','CONNECTION','TOTAL','RATER']
    df_r2 = df[['COMMENTS','EVIDENCE2','SUGGESTION2','CONNECTION2','TOTAL2']].copy()
    df_r2['RATER'] = 2
    df_r2.columns = ['COMMENTS','EVIDENCE','SUGGESTION','CONNECTION','TOTAL','RATER']
    return pd.concat([df_r1, df_r2], ignore_index=True)

train2 = collapse(train_data)
val2   = collapse(val_data)
test2  = collapse(test_data)


Number of Comments: 1509
Evidence Disagreements: 128
Suggestion Disagreements: 101
Connection Disagreements: 130
InterRater Reliability Scores: 0.6925320199553646 0.8441161940822883 0.641542807649718
No. Comments with No Disagreement: 1176
Max Tokens: 391
Train: 915 0.7998251748251748
Validation: 114 0.09965034965034965
Test: 115 0.10052447552447552


In [5]:
# -----------------------------
# DATASET FUNCTIONS
# -----------------------------
class CommentDataset(Dataset):
    def __init__(self, texts, evidence, suggestion, connection, total, tokenizer, max_len=256):
        self.texts = texts
        self.evidence = evidence
        self.suggestion = suggestion
        self.connection = connection
        self.total = total
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        enc = self.tokenizer(
            text, truncation=True, padding="max_length",
            max_length=self.max_len, return_tensors="pt")
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "evidence": torch.tensor(self.evidence[idx], dtype=torch.float),
            "suggestion": torch.tensor(self.suggestion[idx], dtype=torch.float),
            "connection": torch.tensor(self.connection[idx], dtype=torch.float),
            "total": torch.tensor(self.total[idx], dtype=torch.float)}

# -----------------------
# MODEL
# -----------------------
# class OrdinalEvidenceHead(nn.Module):
#     def __init__(self, hidden_size, num_classes=4):
#         super().__init__()
#         self.linear = nn.Linear(hidden_size, num_classes-1)
#     def forward(self, x):
#         logits = self.linear(x)
#         prob = torch.sigmoid(logits)
#         return prob

# def ordinal_class_from_probs(prob):
#     return torch.sum(prob >= 0.5, dim=1)

class ClinicalBERTMultiTaskBinary(nn.Module):
    def __init__(self, model_name=MODEL_NAME, dropout=0.2):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        hidden_size = self.bert.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.evidence_head = nn.Linear(hidden_size, 1)
        self.suggestion_head = nn.Linear(hidden_size,1)
        self.connection_head = nn.Linear(hidden_size,1)

    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:,0,:]
        cls = self.dropout(cls)
        e_logit = self.evidence_head(cls).squeeze(-1)
        s_logit = self.suggestion_head(cls).squeeze(-1)
        c_logit = self.connection_head(cls).squeeze(-1)
        return e_logit, s_logit, c_logit

# -----------------------
# TRAINING
# -----------------------
def train_epoch(model, loader, optimizer, bce_loss_fn, alpha_total=0.5):
    model.train()
    total_loss = 0
    for batch in loader:
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        evidence = batch["evidence"].to(device)
        suggestion = batch["suggestion"].to(device)
        connection = batch["connection"].to(device)
        total_true = batch["total"].to(device)
        e_logit, s_logit, c_logit = model(input_ids, attention_mask)
        loss_e = bce_loss_fn(e_logit, evidence)
        loss_s = bce_loss_fn(s_logit, suggestion)
        loss_c = bce_loss_fn(c_logit, connection)
        e_pred = (torch.sigmoid(e_logit) >= 0.5).float()
        s_pred = torch.sigmoid(s_logit)
        c_pred = torch.sigmoid(c_logit)
        total_pred = e_pred + s_pred + c_pred
        loss_total = nn.MSELoss()(total_pred, total_true)
        loss = loss_e + loss_s + loss_c + alpha_total * loss_total
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

# -----------------------
# EVALUATION
# -----------------------
def eval_model(model, loader):
    model.eval()
    all_e, all_s, all_c, all_total, all_total_true = [], [], [], [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            total_true = batch["total"].to(device)
            e_logit, s_logit, c_logit = model(input_ids, attention_mask)
            e_pred = (torch.sigmoid(e_logit) >= 0.5).cpu().numpy()
            s_pred = (torch.sigmoid(s_logit) >= 0.5).cpu().numpy()
            c_pred = (torch.sigmoid(c_logit) >= 0.5).cpu().numpy()
            total_pred = e_pred + s_pred + c_pred
            all_e.extend(e_pred)
            all_s.extend(s_pred)
            all_c.extend(c_pred)
            all_total.extend(total_pred)
            all_total_true.extend(total_true.cpu().numpy())
    return {
        "evidence_pred": np.array(all_e),
        "suggestion_pred": np.array(all_s),
        "connection_pred": np.array(all_c),
        "total_pred": np.array(all_total),
        "total_true": np.array(all_total_true)}

# -----------------------
# Prepare dataloaders
# -----------------------
train2 = train2[train2["RATER"] == 1]
val2   = val2[val2["RATER"] == 1]
test2  = test2[test2["RATER"] == 1]

train_texts = train2["COMMENTS"].tolist()
train_e = train2["EVIDENCE"].values
train_s = train2["SUGGESTION"].values
train_c = train2["CONNECTION"].values
train_total = train2["TOTAL"].values

val_texts = val2["COMMENTS"].tolist()
val_e = val2["EVIDENCE"].values
val_s = val2["SUGGESTION"].values
val_c = val2["CONNECTION"].values
val_total = val2["TOTAL"].values

test_texts = test2["COMMENTS"].tolist()
test_e = test2["EVIDENCE"].values
test_s = test2["SUGGESTION"].values
test_c = test2["CONNECTION"].values
test_total = test2["TOTAL"].values

train_ds = CommentDataset(train_texts, train_e, train_s, train_c, train_total, tokenizer)
val_ds   = CommentDataset(val_texts, val_e, val_s, val_c, val_total, tokenizer)
test_ds  = CommentDataset(test_texts, test_e, test_s, test_c, test_total, tokenizer)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=16)
test_loader = DataLoader(test_ds, batch_size=16)

# -----------------------
# Initialize model
# -----------------------
model = ClinicalBERTMultiTaskBinary().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
bce_loss_fn = nn.BCEWithLogitsLoss()
EPOCHS = 50

best_val_mse = float("inf")
patience = 10
counter = 0

# -----------------------
# TRAIN
# -----------------------
for epoch in range(EPOCHS):
    train_loss = train_epoch(model, train_loader, optimizer, bce_loss_fn)
    val_results = eval_model(model, val_loader)
    mse_total = mean_squared_error(val_results["total_true"], val_results["total_pred"])
    print(f"Epoch {epoch+1} | Train Loss={train_loss:.4f} | Val Total MSE={mse_total:.4f}")
    # Early stopping
    if mse_total < best_val_mse:
        best_val_mse = mse_total
        counter = 0
        torch.save(model.state_dict(), "/blue/prismap-ai-core/deehu/projects/QUAL_Score_Project/best_model_binary.pt")
    else:
        counter += 1
        if counter >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break
print("\nTraining complete.")

# -----------------------
# TEST EVALUATION
# -----------------------
def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp) if (tn + fp) > 0 else np.nan

def bootstrap_ci(y_true, y_pred, metric_fn, n_bootstraps=2000, ci=95, seed=42):
    rng = np.random.RandomState(seed)
    stats = []
    n = len(y_true)
    indices = np.arange(n)
    for _ in range(n_bootstraps):
        sample_idx = rng.choice(indices, size=n, replace=True)
        stat = metric_fn(y_true[sample_idx], y_pred[sample_idx])
        stats.append(stat)
    stats = np.array(stats)
    lower = np.percentile(stats, (100 - ci) / 2)
    upper = np.percentile(stats, 100 - (100 - ci) / 2)
    return np.mean(stats), lower, upper


BINARY_MODEL_PATH = (
    "/blue/prismap-ai-core/deehu/projects/"
    "QUAL_Score_Project/best_model_binary.pt"
)

model.load_state_dict(
    torch.load(
        BINARY_MODEL_PATH,
        map_location=device,
        weights_only=False
    )
)
model.to(device)
model.eval()

test_results = eval_model(model, test_loader)
mse_test = mean_squared_error(test_results["total_true"], test_results["total_pred"])
print("\nTest Total MSE:", mse_test)


Epoch 1 | Train Loss=1.1601 | Val Total MSE=0.5172
Epoch 2 | Train Loss=0.5571 | Val Total MSE=0.5000
Epoch 3 | Train Loss=0.3048 | Val Total MSE=0.5000
Epoch 4 | Train Loss=0.1560 | Val Total MSE=0.5000
Epoch 5 | Train Loss=0.1005 | Val Total MSE=0.4828
Epoch 6 | Train Loss=0.0738 | Val Total MSE=0.5000
Epoch 7 | Train Loss=0.0596 | Val Total MSE=0.5000
Epoch 8 | Train Loss=0.0540 | Val Total MSE=0.4914
Epoch 9 | Train Loss=0.0386 | Val Total MSE=0.4914
Epoch 10 | Train Loss=0.0336 | Val Total MSE=0.4828
Epoch 11 | Train Loss=0.0438 | Val Total MSE=0.5000
Epoch 12 | Train Loss=0.0319 | Val Total MSE=0.4914
Epoch 13 | Train Loss=0.0391 | Val Total MSE=0.4914
Epoch 14 | Train Loss=0.0231 | Val Total MSE=0.5000
Epoch 15 | Train Loss=0.0501 | Val Total MSE=0.4914
Early stopping at epoch 15

Training complete.

Test Total MSE: 0.5966386795043945


In [6]:
# ==================================
# EVIDENCE (BINARY)
# ==================================
y_true_e = test_ds.evidence
# y_pred_e = (torch.tensor(test_results["evidence_pred"]) >= 0.5).numpy()  # threshold 0.5
y_pred_e = test_results["evidence_pred"].astype(int)


acc_e, acc_e_l, acc_e_u = bootstrap_ci(y_true_e, y_pred_e, accuracy_score)
f1_e,  f1_e_l, f1_e_u  = bootstrap_ci(y_true_e, y_pred_e, f1_score)
sens_e, sens_e_l, sens_e_u = bootstrap_ci(y_true_e, y_pred_e, recall_score)
spec_e, spec_e_l, spec_e_u = bootstrap_ci(y_true_e, y_pred_e, specificity_score)

print("\nEVIDENCE (Binary)")
print(f"Accuracy:     {acc_e:.3f} (95% CI {acc_e_l:.3f}–{acc_e_u:.3f})")
print(f"F1 Score:     {f1_e:.3f} (95% CI {f1_e_l:.3f}–{f1_e_u:.3f})")
print(f"Sensitivity:  {sens_e:.3f} (95% CI {sens_e_l:.3f}–{sens_e_u:.3f})")
print(f"Specificity:  {spec_e:.3f} (95% CI {spec_e_l:.3f}–{spec_e_u:.3f})")

# ==================================
# SUGGESTION (BINARY)
# ==================================
y_true_s = test_ds.suggestion
y_pred_s = test_results["suggestion_pred"]

acc_s, acc_s_l, acc_s_u = bootstrap_ci(y_true_s, y_pred_s, accuracy_score)
f1_s, f1_s_l, f1_s_u = bootstrap_ci(y_true_s, y_pred_s, lambda yt, yp: f1_score(yt, yp))
sens_s, sens_s_l, sens_s_u = bootstrap_ci(y_true_s, y_pred_s, recall_score)
spec_s, spec_s_l, spec_s_u = bootstrap_ci(y_true_s, y_pred_s, specificity_score)

print("\nSUGGESTION (BINARY)")
print(f"Accuracy:    {acc_s:.3f} (95% CI {acc_s_l:.3f}–{acc_s_u:.3f})")
print(f"F1 Score:    {f1_s:.3f} (95% CI {f1_s_l:.3f}–{f1_s_u:.3f})")
print(f"Sensitivity: {sens_s:.3f} (95% CI {sens_s_l:.3f}–{sens_s_u:.3f})")
print(f"Specificity: {spec_s:.3f} (95% CI {spec_s_l:.3f}–{spec_s_u:.3f})")

# ==================================
# CONNECTION (BINARY)
# ==================================
y_true_c = test_ds.connection
y_pred_c = test_results["connection_pred"]

acc_c, acc_c_l, acc_c_u = bootstrap_ci(y_true_c, y_pred_c, accuracy_score)
f1_c, f1_c_l, f1_c_u = bootstrap_ci(y_true_c, y_pred_c, lambda yt, yp: f1_score(yt, yp))
sens_c, sens_c_l, sens_c_u = bootstrap_ci(y_true_c, y_pred_c, recall_score)
spec_c, spec_c_l, spec_c_u = bootstrap_ci(y_true_c, y_pred_c, specificity_score)

print("\nCONNECTION (BINARY)")
print(f"Accuracy:    {acc_c:.3f} (95% CI {acc_c_l:.3f}–{acc_c_u:.3f})")
print(f"F1 Score:    {f1_c:.3f} (95% CI {f1_c_l:.3f}–{f1_c_u:.3f})")
print(f"Sensitivity: {sens_c:.3f} (95% CI {sens_c_l:.3f}–{sens_c_u:.3f})")
print(f"Specificity: {spec_c:.3f} (95% CI {spec_c_l:.3f}–{spec_c_u:.3f})")



EVIDENCE (Binary)
Accuracy:     0.908 (95% CI 0.857–0.958)
F1 Score:     0.946 (95% CI 0.912–0.976)
Sensitivity:  0.942 (95% CI 0.893–0.981)
Specificity:  0.705 (95% CI 0.476–0.917)

SUGGESTION (BINARY)
Accuracy:    0.975 (95% CI 0.941–1.000)
F1 Score:    0.943 (95% CI 0.867–1.000)
Sensitivity: 0.961 (95% CI 0.875–1.000)
Specificity: 0.979 (95% CI 0.945–1.000)

CONNECTION (BINARY)
Accuracy:    0.958 (95% CI 0.916–0.992)
F1 Score:    0.793 (95% CI 0.571–0.952)
Sensitivity: 0.713 (95% CI 0.455–0.933)
Specificity: 0.991 (95% CI 0.971–1.000)
